# Readme E-Manuscripta

Dieses Jupyter Notebook bereitet Datenobjekte und Metadaten aus dem E-Manuscripta-Bestand vor für die digitale Langzeitarchivierung (DLZA) der ZHB Luzern, basierend auf der GOCFL implementierung von Jürgen Enge, https://github.com/je4/gocfl .

Mit Stand vom Juli 2023 gibt es 22 Objekte in E-Manuscripta der ZHB. Aufgrund der geringen Menge und der diversen Metadatenquellen, wird eine Excel-Datei händisch ergänzt mit den in Alma fehlenden Daten (bspw. DOI, url). 


## Basiskonfiguration

Die E-Manuscripta-Signaturen setzen sich aus dem E-Manuscripta DOI zusammen. Im Config-File wird die Abteilung hinzugefügt, das Script ergänzt den DOI. 


### Config.py

Beispieldaten für ZHB E-Manuscripta. Anpassungen können in der config.py vorgenommen werden. 

    address = 'mailto:someone@internet.com'
    collection = 'ZHB E-Manuscripta'
    collection_id = 'zhb_emanuscripta'
    last_changed = 'yyy-mm-dd'
    organisation = 'Zentral- und Hochschulbibliothek Luzern'
    organisation_id = 'zhb'
    signature = 'zhb_'
    

### Import file

Die E-Manuscripta werden aus der Zenodo-OAI-Schnittstelle abgeholt. Sie befinden sich in folgender community:
https://zenodo.org/communities/lara_e-manuscripta

Weitere Infos: https://developers.zenodo.org/#oai-pmh

Alternativ zur OAI-Abfrage existiert auch eine Excel-Datei, in welcher die HAN-Nummern abgelegt sind, die in vielen Fällen für die Dateibezeichnungen verwendet werden. Diese Datei kann mit wenig manuellem Aufwand aus Alma extrahiert werden. Die E-Manuscripta befinden sich im Alma-Set e-manuscripta_dlza_heka. Sie wird hier nicht verwendet. 


### Export

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format info/signature.json.
Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert (directory 'fulldump').

### Marcxml aus Alma (SRU)

Mit der alma_id werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert unter signature.xml
Zusätzlich wird die HAN-Nummer aus den Alma-Daten extrahiert, welche für die Objekt-Bezeichnung (Bennennung der ZIP-Kapseln) wichtig ist. 


### TODO Datenobjekte abholen
Die E-Manuscripta-Zipkapseln sind alle nach folgender Struktur benannt:

    HAN-Nummer _ (Digitalisierungsdatum/Workflow?) _ master _ version . zip

Beispiel:

    000218785_20150312T000312_master_ver1.zip

Ausnahme: 
https://dx.doi.org/10.7891/e-manuscripta-108732

"Der ächte Eidsgenoss, eine wöchtentliche Sittenschrift. Zweyter Jahrgang / hrsg. von Johann Jakob Spreng"
Dateiname:

    10_7891_e-manuscripta-108732.zip 

In diesem Script wird vorausgesetzt, dass die Daten mit dieser Bezeichnung im Ordner objects liegen, da die Dateibenennung nicht einheitlich ist. 

### TODO create Befehle erstellen für gocfl 

Die gocfl create Befehle für die E-Manuscripta-Sammlung (alle Objekte) werden gemäss https://github.com/je4/gocfl/blob/main/docs/create.md erstellt mit der Signature. 

Muster:

    gocfl create P:/temp/archiv P:/temp/testdata/zhb_erara_86774/object metadata:p:/temp/testdata/zhb_erara_86774/metadata --config p:/temp/config/gocfl.toml -i "zhb_erara_86774" --ext-NNNN-metafile-source p:/temp/testdata/zhb_erara_86774.json 

In [3]:
from sickle import Sickle
import json
import config
import xml.etree.ElementTree as ET
import requests
import pandas as pd
import re

# Initialize the client by passing the base URL and fetch records from the OAI Set

base_url = 'https://zenodo.org/oai2d'
prefix = 'oai_dc'
set_name = 'user-lara_e-manuscripta'

sickle = Sickle(base_url)
records = sickle.ListRecords(metadataPrefix=prefix, set=set_name)
record = records.next()

completed_iterating = False
recordCount = 1
completeSet = []

# Iterate through records and collect metadata info for json export
# Uncomment one of the next 2 lines for testing/production:

while not completed_iterating:
#while recordCount < 5:
    try:
        
        infoSet = {}
                
        infoSet["additional"] = ''
        infoSet["address"] = config.address
        infoSet["collection"] = config.collection
        infoSet["collection_id"] = config.collection_id
        infoSet["created"] = record.metadata["date"][0]
        infoSet["identifiers"] = record.metadata["identifier"]
        infoSet["ingest_workflow"] = config.ingest_workflow
        infoSet["keywords"] = config.keywords
        infoSet["last_changed"] = config.last_changed
        infoSet["organisation"] = config.organisation
        infoSet["organisation_id"] = config.organisation_id
        infoSet["references"] = record.metadata["relation"]
        infoSet["sets"] = config.sets
        infoSet["signature"] = ''
        infoSet["title"] = record.metadata["title"][0]
        infoSet["user"] = record.metadata["creator"][0]
        
        # add e-Manuscripta doi to identifiers:
        doi = infoSet["references"][0][4:]  #        print(doi)
        infoSet["identifiers"].insert(0, doi)
        
        # add alma id to identifiers:
        alma_link = infoSet["references"][1]
        alma_id = alma_link.partition('alma')[2]
        infoSet["identifiers"].insert(0, str(alma_id)) #        print(alma_id)  
        
        # create signature
        signature = doi.replace('.','_').replace('/','_')    
        signature = config.signature+signature
        infoSet["signature"] = signature        #print(signature)
        
        # get metadata from Alma OAI as MARCXML        
        sru_url = "https://slsp-rzs.alma.exlibrisgroup.com/view/sru/41SLSP_RZS"
        query = f"{sru_url}?version=1.2&operation=searchRetrieve&recordSchema=marcxml&query=rec.id={alma_id}"
        #print(query)
        response = requests.get(query)
        if response.status_code != 200:
            raise Exception(f"SRU request failed with status code {response.status_code}")
        
        # get HAn Number from field 035, eg.: (HAN)000218785DSV05
        rec_string = response.text
        han_nr_list = re.findall('\(HAN\)\d{9}DSV05', rec_string)
        han_nr = ''
        if len(han_nr_list):
            han_nr = han_nr_list[0]        #print(han_nr)
        
        # add HAN number to identifiers, additional without (HAN+DSV05), as this is part of the file name        
        infoSet["identifiers"].insert(0, han_nr)
        if han_nr != '':
            infoSet["additional"] = str(han_nr[5:14])
        else:
            infoSet["additional"] = signature
        
        #debugging: print(infoSet)
        
        # prepare filename for json export
        info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
        infofile = f"info/{signature}.json"
        with open(infofile, "w") as outfile:
            outfile.write(info_json)
            print(f"info.json saved as {infofile}")
        
        # add info to completeSet
        completeSet.append(infoSet)       

        # Save the response content (MARCXML) to a file
        metafile = f"metadata/{signature}.xml"
        with open(metafile, 'wb') as file:
            file.write(response.content)
            print(f"Record with ID {alma_id} saved as {metafile}")    

        #continue with next record
        recordCount = recordCount +1
        record = records.next()
        
    except StopIteration:
        completed_iterating = True

# Writing completeSet as json file
fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)
fulljsonfile = "fulldump/emanuscripta_complete_set.json"
with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nfulldump written to {fulljsonfile}")
    
# Writing completeSet as Excel file
fullexcelfile = "fulldump/emanuscripta_complete_set.xlsx"
df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"Saved fulldump in excel file as {fullexcelfile}")


info.json saved as info/zhb_10_7891_e-manuscripta-23742.json
Record with ID 9914249311805505 saved as metadata/zhb_10_7891_e-manuscripta-23742.xml
info.json saved as info/zhb_10_7891_e-manuscripta-108732.json
Record with ID 9914289852805505 saved as metadata/zhb_10_7891_e-manuscripta-108732.xml
info.json saved as info/zhb_10_7891_e-manuscripta-25217.json
Record with ID 9914249311205505 saved as metadata/zhb_10_7891_e-manuscripta-25217.xml
info.json saved as info/zhb_10_7891_e-manuscripta-24545.json
Record with ID 9914249332505505 saved as metadata/zhb_10_7891_e-manuscripta-24545.xml
info.json saved as info/zhb_10_7891_e-manuscripta-25384.json
Record with ID 9914249314005505 saved as metadata/zhb_10_7891_e-manuscripta-25384.xml
info.json saved as info/zhb_10_7891_e-manuscripta-24480.json
Record with ID 9914249313905505 saved as metadata/zhb_10_7891_e-manuscripta-24480.xml
info.json saved as info/zhb_10_7891_e-manuscripta-23891.json
Record with ID 9914249311105505 saved as metadata/zhb_1